# Lung Segmentation with U-Net (ResNet34 Backbone)
COVID-QU-Ex Dataset

In [ ]:
!pip install -q albumentations segmentation-models-pytorch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Setup & Data Preparation

In [ ]:
import os, glob, random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
!cp /content/drive/MyDrive/Data_3/data.zip /content/
!unzip -q /content/data.zip -d /content/data
!ls /content/data

In [ ]:
DATA_ROOT = "/content/data/Lung Segmentation Data/Lung Segmentation Data"
IMG_SIZE = 256
BATCH_SIZE = 32
LR = 5e-4
EPOCHS = 25
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
IMG_EXTS = (".png", ".jpg", ".jpeg")

CLASSES = sorted([
    d for d in os.listdir(os.path.join(DATA_ROOT, "Train"))
    if os.path.isdir(os.path.join(DATA_ROOT, "Train", d))
])
print("Classes:", CLASSES)

def get_mask_path(img_path):
    return img_path.replace(os.sep + "images", os.sep + "lung masks")

def load_pairs(split):
    pairs = []
    for cls in CLASSES:
        img_dir = os.path.join(DATA_ROOT, split, cls, "images")
        images = glob.glob(os.path.join(img_dir, "**", "*.*"), recursive=True)
        images = [p for p in images if p.lower().endswith(IMG_EXTS)]
        for ip in images:
            mp = get_mask_path(ip)
            if os.path.exists(mp):
                pairs.append((ip, mp))
    print(f"[{split}] {len(pairs)} pairs")
    return pairs

train_pairs = load_pairs("Train")
val_pairs = load_pairs("Val")
test_pairs = load_pairs("Test")

## 2. EDA

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(3):
    idx = random.randint(0, len(train_pairs) - 1)
    img_path, mask_path = train_pairs[idx]
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    overlay = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    overlay[mask > 127, 1] = 200
    
    axes[i, 0].imshow(img, cmap='gray'); axes[i, 0].set_title('Image'); axes[i, 0].axis('off')
    axes[i, 1].imshow(mask, cmap='gray'); axes[i, 1].set_title('Lung Mask'); axes[i, 1].axis('off')
    axes[i, 2].imshow(overlay); axes[i, 2].set_title('Overlay'); axes[i, 2].axis('off')
    axes[i, 3].hist(img.ravel(), bins=50, color='steelblue'); axes[i, 3].set_title('Histogram')

plt.tight_layout()
plt.show()

In [ ]:
split_counts = {'Train': len(train_pairs), 'Val': len(val_pairs), 'Test': len(test_pairs)}
plt.figure(figsize=(8, 4))
plt.bar(split_counts.keys(), split_counts.values(), color=['#3498db', '#2ecc71', '#e74c3c'])
plt.title('Dataset Split Distribution')
plt.ylabel('Number of Samples')
for k, v in split_counts.items():
    plt.text(k, v + 100, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Dataset & Augmentations

In [ ]:
train_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.3),
    ToTensorV2(),
])

val_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    ToTensorV2(),
])

class LungDataset(Dataset):
    def __init__(self, pairs, transforms):
        self.pairs = pairs
        self.transforms = transforms

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        img_path, mask_path = self.pairs[i]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        img = img.astype(np.float32) / 255.0
        mask = (mask > 127).astype(np.float32)

        augmented = self.transforms(image=img, mask=mask)
        x = augmented['image'].float()
        y = augmented['mask'].float()

        if x.ndim == 2: x = x.unsqueeze(0)
        if y.ndim == 2: y = y.unsqueeze(0)
        return x, y

train_ds = LungDataset(train_pairs, train_aug)
val_ds = LungDataset(val_pairs, val_aug)
test_ds = LungDataset(test_pairs, val_aug)

nw = min(4, os.cpu_count())
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=nw, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=nw, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=nw, pin_memory=True)

print(f"Train: {len(train_loader)} batches | Val: {len(val_loader)} batches | Test: {len(test_loader)} batches")

## 4. Model

In [ ]:
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=1,
    classes=1,
    activation=None
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 5. Loss & Metrics

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()

def dice_score(pred, target, eps=1e-6):
    pred_bin = (pred > 0.5).float()
    intersection = (pred_bin * target).sum(dim=(2, 3))
    union = pred_bin.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return ((2 * intersection + eps) / (union + eps)).mean().item()

def iou_score(pred, target, eps=1e-6):
    pred_bin = (pred > 0.5).float()
    intersection = (pred_bin * target).sum(dim=(2, 3))
    union = pred_bin.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) - intersection
    return ((intersection + eps) / (union + eps)).mean().item()

def combined_loss(logits, target):
    probs = torch.sigmoid(logits)
    inter = (probs * target).sum(dim=(2, 3))
    union = probs.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    dice_l = 1.0 - ((2 * inter + 1e-6) / (union + 1e-6)).mean()
    return bce_loss(logits, target) + dice_l

## 6. Training

In [ ]:
from torch.amp import GradScaler, autocast

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = GradScaler('cuda', enabled=(DEVICE == 'cuda'))

def train_one_epoch(loader):
    model.train()
    total_loss, total_dice, total_iou = 0.0, 0.0, 0.0
    for x, y in tqdm(loader, leave=False, desc='Train'):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with autocast('cuda', enabled=(DEVICE == 'cuda')):
            logits = model(x)
            loss = combined_loss(logits, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        probs = torch.sigmoid(logits).detach()
        total_loss += loss.item()
        total_dice += dice_score(probs, y)
        total_iou += iou_score(probs, y)
    n = len(loader)
    return total_loss / n, total_dice / n, total_iou / n

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, total_dice, total_iou = 0.0, 0.0, 0.0
    for x, y in tqdm(loader, leave=False, desc='Eval'):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with autocast('cuda', enabled=(DEVICE == 'cuda')):
            logits = model(x)
            loss = combined_loss(logits, y)
        probs = torch.sigmoid(logits)
        total_loss += loss.item()
        total_dice += dice_score(probs, y)
        total_iou += iou_score(probs, y)
    n = len(loader)
    return total_loss / n, total_dice / n, total_iou / n

In [ ]:
history = {'tr_loss': [], 'va_loss': [], 'tr_dice': [], 'va_dice': [], 'tr_iou': [], 'va_iou': [], 'lr': []}
best_dice = 0.0
patience, wait = 7, 0
best_path = 'best_lung_seg.pth'

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_dice, tr_iou = train_one_epoch(train_loader)
    va_loss, va_dice, va_iou = evaluate(val_loader)
    cur_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    history['tr_loss'].append(tr_loss); history['va_loss'].append(va_loss)
    history['tr_dice'].append(tr_dice); history['va_dice'].append(va_dice)
    history['tr_iou'].append(tr_iou); history['va_iou'].append(va_iou)
    history['lr'].append(cur_lr)

    print(f"Epoch {epoch:02d} | lr {cur_lr:.2e} | "
          f"train loss {tr_loss:.4f} dice {tr_dice:.4f} iou {tr_iou:.4f} | "
          f"val loss {va_loss:.4f} dice {va_dice:.4f} iou {va_iou:.4f}")

    if va_dice > best_dice + 1e-4:
        best_dice = va_dice
        wait = 0
        torch.save(model.state_dict(), best_path)
        print(f"  -> saved (best dice: {best_dice:.4f})")
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(torch.load(best_path, map_location=DEVICE))
print(f"\nLoaded best model (val_dice = {best_dice:.4f})")

## 7. Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['tr_loss'], label='Train')
axes[0].plot(history['va_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(history['tr_dice'], label='Train')
axes[1].plot(history['va_dice'], label='Val')
axes[1].set_title('Dice Score'); axes[1].legend(); axes[1].set_xlabel('Epoch')

axes[2].plot(history['tr_iou'], label='Train')
axes[2].plot(history['va_iou'], label='Val')
axes[2].set_title('IoU Score'); axes[2].legend(); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

## 8. Test Evaluation

In [ ]:
test_loss, test_dice, test_iou = evaluate(test_loader)
print(f"Test Results: loss={test_loss:.4f} | dice={test_dice:.4f} | iou={test_iou:.4f}")

## 9. Predictions

In [ ]:
model.eval()
fig, axes = plt.subplots(4, 3, figsize=(14, 18))

for row in range(4):
    idx = random.randint(0, len(test_ds) - 1)
    x, y = test_ds[idx]
    with torch.no_grad():
        logits = model(x.unsqueeze(0).to(DEVICE))
        pred = (torch.sigmoid(logits).cpu()[0, 0].numpy() > 0.5).astype('float32')

    img = x[0].numpy()
    gt = y[0].numpy()

    axes[row, 0].imshow(img, cmap='gray'); axes[row, 0].set_title('Input'); axes[row, 0].axis('off')
    axes[row, 1].imshow(gt, cmap='gray'); axes[row, 1].set_title('Ground Truth'); axes[row, 1].axis('off')
    axes[row, 2].imshow(pred, cmap='gray'); axes[row, 2].set_title('Prediction'); axes[row, 2].axis('off')

plt.tight_layout()
plt.show()